# Job Category Chatbot
My app helps a job board manager decide which country and sector to target for a first category-repair campaign by comparing missing tags and AI-labeled job titles across eight countries.
1. Where do job postings have the most missing categories?
2. How often did Gemini agree with comparable tags, and which sectors were hidden in the missing tags?
3. Which sectors lead after filling the gaps, and where should the manager run a pilot? This is my original sub-question.

## 1. Read the two public CSV files
Each line in these files is a row. The helper keeps quoted commas and literal quotation marks inside fields, such as the comma in “accounting, banking & finance.”

In [1]:
import pandas as pd
import requests

def csv_fields(line):
    fields = [""]
    quoted = False
    i = 0
    while i < len(line):
        char = line[i]
        if quoted and line[i:i+2] == '""':
            fields[-1] += '"'
            i += 1
        elif char == '"':
            quoted = not quoted
        elif char == "," and not quoted:
            fields.append("")
        else:
            fields[-1] += char
        i += 1
    return fields

In [2]:
def load_csv(filename):
    base = "https://raw.githubusercontent.com/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/main/"
    lines = requests.get(base + filename).text.strip().split("\n")
    columns = {}
    for name in csv_fields(lines[0].strip()):
        columns[name] = []
    for line in lines[1:]:
        values = csv_fields(line.strip())
        for i, name in enumerate(columns):
            columns[name].append(values[i])
    return pd.DataFrame(columns)

jobs = load_csv("dfall_clean.csv")
results = load_csv("completed_results.csv")
print("Postings loaded:", len(jobs))

Postings loaded: 4000


## 2. Count missing categories by country
Both source tags below count as missing. Countries are ranked by their missing percentage.

In [3]:
def is_missing(tag):
    return tag in ["unknown", "other-general-jobs"]

def missing_by_country(jobs):
    flags = jobs["category_tag"].apply(is_missing)
    totals = {}
    missing = {}
    for i, country in enumerate(jobs["country"]):
        totals[country] = totals.get(country, 0) + 1
        missing[country] = missing.get(country, 0) + int(flags[i])
    rates = {}
    for country in totals:
        rates[country] = 100 * missing[country] / totals[country]
    ranked = sorted(rates, key=rates.get, reverse=True)
    lines = []
    for country in ranked:
        lines.append(f"{country}: {missing[country]}/{totals[country]} ({round(rates[country], 1)}%)")
    return {"priority": ranked[0], "text": "\n".join(lines)}

audit = missing_by_country(jobs)
print(audit["text"])

Germany: 434/500 (86.8%)
Switzerland: 355/500 (71.0%)
France: 325/500 (65.0%)
Spain: 323/500 (64.6%)
Netherlands: 281/500 (56.2%)
Italy: 159/500 (31.8%)
United Kingdom: 2/500 (0.4%)
USA: 0/500 (0.0%)


## 3. Give an originally missing title an AI sector
The earlier full run used job titles and country to choose one of 16 sectors. Its public totals power the full analysis. This small UK example replays a real Colab result by default.
For a fresh label and a free-text chatbot answer, enable notebook access to GOOGLE_API_KEY in Colab Secrets and set RUN_LIVE to True.

In [4]:
class JobChatbot:
    def __init__(self, sectors):
        self.sectors = sectors
        self.evidence = ""

    def clean_label(self, answer):
        label = answer.strip().lower()
        if label in self.sectors:
            return label
        return "uncertain"

    def ask(self, question):
        prompt = self.evidence + "\n" + question
        reply = self.client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)
        return reply.text.strip()

    def label_title(self, title):
        choices = ", ".join(self.sectors)
        prompt = f"UK job title: {title}. Return one sector from this list, or uncertain: {choices}"
        return self.clean_label(self.ask(prompt))

In [5]:
RUN_LIVE = False
sectors = []
for i, part in enumerate(results["part"]):
    if part == "hidden":
        sectors.append(results["sector"][i])
chatbot = JobChatbot(sectors)
demo = pd.DataFrame({"title": ["Teaching Assistant"]})
demo["ai_type"] = [chatbot.clean_label("teaching & education")]
if RUN_LIVE:
    from google.colab import userdata
    from google import genai
    chatbot.client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
    demo["ai_type"] = demo["title"].apply(chatbot.label_title)
print(demo)

                title               ai_type
0  Teaching Assistant  teaching & education


## 4. Check agreement and reveal hidden sectors
The full run compared 2,093 original labels mapped to the 16 sectors. Two received uncertain AI labels, leaving 2,091 comparable pairs with an AI sector.

In [6]:
def agreement_and_hidden(results, demo):
    checked = int(float(results["checked"][0]))
    matches = int(float(results["matches"][0]))
    filled = int(float(results["filled"][0]))
    missing = int(float(results["missing"][0]))
    rate = round(100 * matches / checked, 1)
    lines = [f"Agreement: {matches}/{checked} ({rate}%)", f"Missing categories filled: {filled}/{missing}"]
    for i, part in enumerate(results["part"]):
        if part == "hidden":
            count = int(float(results["count"][i]))
            lines.append(f"{results['sector'][i]}: {count}")
    lines.append(f"UK title check: {demo['title'][0]} → {demo['ai_type'][0]}")
    return "\n".join(lines)

check = agreement_and_hidden(results, demo)
print(check)

Agreement: 1355/2091 (64.8%)
Missing categories filled: 1876/1879
trade & construction: 407
logistics, driver & supply chain: 227
engineering: 204
accounting, banking & finance: 176
sales & business development: 157
healthcare & nursing: 132
maintenance, cleaning & facility: 122
it & software development: 102
other / specialized: 97
hospitality & catering: 63
administrative & office support: 52
customer service & support: 44
hr & recruitment: 27
social work & care: 27
teaching & education: 24
marketing, pr & advertising: 15
UK title check: Teaching Assistant → teaching & education


## 5. Compare sector rankings and choose a pilot
The public shift rows contain the three largest final sectors per country, with alphabetical tie-breaking. The full run kept known source labels and filled only missing labels with valid AI sectors.

In [7]:
def sector_shifts(results, priority, demo):
    lines = []
    leaders = {}
    for i, part in enumerate(results["part"]):
        if part == "shift":
            country = results["country"][i]
            sector = results["sector"][i]
            before = int(float(results["before"][i]))
            after = int(float(results["after"][i]))
            lines.append(f"{country} / {sector}: {before} → {after} (+{after - before})")
            if country not in leaders:
                leaders[country] = sector
            if country == "United Kingdom" and sector == "teaching & education":
                uk_education = before
    gain = int(demo["ai_type"][0] == "teaching & education")
    lines.append(f"UK one-title scenario: education {uk_education} → {uk_education + gain} (+{gain})")
    lines.append(f"Decision: pilot category repair in {priority} and test a campaign for {leaders[priority]} after a label review.")
    return "\n".join(lines)

shifts = sector_shifts(results, audit["priority"], demo)
print(shifts)

France / healthcare & nursing: 128 → 156 (+28)
France / accounting, banking & finance: 3 → 105 (+102)
France / trade & construction: 0 → 61 (+61)
Germany / engineering: 6 → 81 (+75)
Germany / healthcare & nursing: 9 → 73 (+64)
Germany / trade & construction: 0 → 67 (+67)
Italy / other / specialized: 117 → 123 (+6)
Italy / logistics, driver & supply chain: 44 → 52 (+8)
Italy / sales & business development: 12 → 52 (+40)
Netherlands / logistics, driver & supply chain: 50 → 118 (+68)
Netherlands / trade & construction: 7 → 63 (+56)
Netherlands / other / specialized: 35 → 52 (+17)
Spain / trade & construction: 0 → 114 (+114)
Spain / it & software development: 31 → 83 (+52)
Spain / sales & business development: 49 → 73 (+24)
Switzerland / trade & construction: 65 → 162 (+97)
Switzerland / engineering: 12 → 81 (+69)
Switzerland / logistics, driver & supply chain: 6 → 51 (+45)
USA / healthcare & nursing: 179 → 179 (+0)
USA / hospitality & catering: 67 → 67 (+0)
USA / customer service & suppor

## 6. Ask the chatbot about the results
The default prints the answer from Q1. With RUN_LIVE enabled, the same question goes to Gemini with the Q1–Q3 evidence.

In [8]:
question = "What is the country with the most missing categories?"
chatbot.evidence = "Answer only from these project results.\n" + audit["text"] + "\n" + check + "\n" + shifts
print(question)
if RUN_LIVE:
    print(chatbot.ask(question))
else:
    print("From Q1:", audit["text"].split("\n")[0])

What is the country with the most missing categories?
From Q1: Germany: 434/500 (86.8%)


## Reading the results
Agreement compares Gemini with existing tags. A reviewer would need to label titles independently to measure accuracy.
The eight countries each contribute 500 postings. A before-and-after increase counts repaired tags within this sample. A campaign pilot would measure any effect on applications separately.